In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import pint
from imagematerials.util import dataset_to_array

In [2]:
target_regions= ["Canada",
"Central Europe",
"China",
"India",
"Japan",
"USA",
"Western Europe",
"Latin America",
"Middle East and Northern Africa",
"Other Asia",
"Other OECD",
"Reforming Economies"
"Subsaharan Africa"]

target_variables_narrow = {"Total pass. travel demand":"p-km/cap",
                    "Total freight. travel demand":"t-km/cap",
                    "passenger car":"Modal split - pkm/capita",
                    "passenger bus":"Modal split - pkm/capita",
                    "passenger aviation":"Modal split - pkm/capita",
                    "passenger walk":"Modal split - %",
                    "passenger bike":"Modal split - %",
                    "freight road heavy":"Modal split - tkm/capita",
                    "freight road light":"Modal split - tkm/capita",
                    "freight rail":"Modal split - tkm/capita",
                    "freight aviation":"Modal split - tkm/capita",
                    "Numb passengers per private vehicle": "Occupancy rate",
                    "Numb passengers per BUS": "Occupancy rate",
                    "Numb passengers per train": "Occupancy rate",
                    "Numb passengers per aviation passengers": "Occupancy rate",
	                "Ton per heavy freight": "Load rate",
                    "Ton per light freight": "Load rate",
                    "Ton per aviation freight": "Load rate",
                    "Ton per navigation freight": "Load rate"
}

target_variables_slow = {
                    "car": "kg/unit",
                    "bus": "kg/unit",
                    "truck": "kg/unit",
                    "airplane": "kg/unit",
                    "rail": "kg/unit",
                    "ships": "kg/unit",
                    "bicycle": "kg/unit",
}

all_variables = list(target_variables_narrow.keys()) + list(target_variables_slow.keys())

# Create the xarray dataset

vehicle_targets = xr.DataArray(
            0.0,
            dims=("Time", "Region", "variable"),
            coords={"Time": [2019,2060],
                    "Region": target_regions,
                    "variable": all_variables})



In [18]:
image_output = pd.read_csv("IMAGE scenario explorer variables.csv", delimiter=";", index_col=0, usecols=range(8))
image_output = image_output.drop(columns=["model", "scenario","unit"])

# Step 2: Convert DataFrame into xarray Dataset
xr_image_output = image_output.set_index(["variable", "region", "year"]).to_xarray()



xr_image_output = xr_image_output.sel(year=[2020, 2060])

xr_image_output = xr_image_output.to_dataarray("var2").drop_vars("var2")

xr_image_output.squeeze()


<xarray.DataArray (variable: 245, region: 42, year: 2)> Size: 165kB
array([[[2.51986672e+03, 6.05742459e+03],
        [1.75463998e+04, 2.36588462e+04],
        [1.19312867e+04, 9.64523157e+03],
        ...,
        [2.41391315e+03, 2.02927707e+03],
        [3.32774785e+03, 5.44567340e+03],
        [3.74342461e+04, 5.04594206e+04]],

       [[4.31039706e+03, 9.37262323e+03],
        [2.30384006e+04, 3.09683914e+04],
        [1.45787748e+04, 1.25057313e+04],
        ...,
        [3.57161432e+03, 3.69490015e+03],
        [4.40325500e+03, 6.76035303e+03],
        [5.21122695e+04, 7.06829219e+04]],

       [[2.84213496e+00, 1.61457678e+01],
        [1.78500915e+01, 7.47940536e+01],
        [1.13828537e+01, 2.93207886e+01],
        ...,
...
        ...,
        [7.81019993e+01, 8.74366236e+01],
        [2.76977005e+02, 4.20949905e+02],
        [4.21509424e+03, 4.62572314e+03]],

       [[1.82954300e-03, 9.65382100e-03],
        [1.52827860e-02, 4.67524630e-02],
        [9.97412000e-03, 2.05959830e-02],
        ...,
        [1.08348800e-03, 2.08376200e-03],
        [2.84921000e-03, 9.55518200e-03],
        [3.40284930e-02, 9.13541720e-02]],

       [[3.24283500e-03, 2.18385510e-02],
        [2.40310800e-02, 9.72339040e-02],
        [1.49312390e-02, 4.20886500e-02],
        ...,
        [2.04756400e-03, 4.32523100e-03],
        [4.09839100e-03, 1.62562540e-02],
        [7.39214800e-02, 2.19615613e-01]]])
Coordinates:
  * variable  (variable) object 2kB 'Emissions|CO2' ... 'Value Added|Services'
  * region    (region) object 336B 'Africa (R10)' 'Asia (R5)' ... 'World'
  * year      (year) int64 16B 2020 2060

In [9]:
# Mapping from xr_image_output regions to target regions
region_mapping = {
    "IMAGE 3.4|Canada": "Canada",
    "IMAGE 3.4|Central Europe": "Central Europe",
    "IMAGE 3.4|China": "China",
    "IMAGE 3.4|India": "India",
    "IMAGE 3.4|Japan": "Japan",
    "IMAGE 3.4|USA": "USA",
    "IMAGE 3.4|Western Europe": "Western Europe",
    "Latin America (R10)": "Latin America",
    "Latin America (R5)": "Latin America",
    "IMAGE 3.4|Middle East": "Middle East and Northern Africa",
    "IMAGE 3.4|Northern Africa": "Middle East and Northern Africa",
    "Rest of Asia (R10)": "Other Asia",
    "IMAGE 3.4|Indonesia": "Other Asia",
    "IMAGE 3.4|Korea": "Other OECD",
    "OECD & EU (R5)": "Other OECD",
    "Reforming Economies (R10)": "Reforming Economies",
    "Reforming Economies (R5)": "Reforming Economies",
    "IMAGE 3.4|Rest of Southern Africa": "Subsaharan Africa",
    "IMAGE 3.4|Western Africa": "Subsaharan Africa",
    "IMAGE 3.4|Eastern Africa": "Subsaharan Africa"
}

# Convert the region names in xr_image_output to match the target regions
xr_regions = xr_image_output.coords["region"].values  # Extract current region names

# Create a mapping of only existing regions
region_mapping_filtered = {key: val for key, val in region_mapping.items() if key in xr_regions}

# Map the dataset’s regions to the target regions
xr_image_output_filtered = xr_image_output.sel(region=list(region_mapping_filtered.keys()))
xr_image_output_filtered = xr_image_output_filtered.assign_coords(region=[region_mapping_filtered[r] for r in xr_image_output_filtered["region"].values])

# Aggregate by summing over the mapped regions
xr_image_output_aggregated = xr_image_output_filtered.groupby("region").sum()
xr_image_output_aggregated



<xarray.Dataset> Size: 53kB
Dimensions:   (region: 13, variable: 245, year: 2)
Coordinates:
  * variable  (variable) object 2kB 'Emissions|CO2' ... 'Value Added|Services'
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Western Europe'
Data variables:
    value     (region, variable, year) float64 51kB 623.6 525.6 ... 0.02182

In [10]:
# Step 3: Extract relevant variables
pop = xr_image_output_aggregated.sel(variable="Population")  # Extract Population data
energy_service = xr_image_output_aggregated.sel(variable=[v for v in xr_image_output_aggregated["variable"].values if "Energy Service|Transportation|" in v or "Product Stock|Transportation|" in v])


energy_service_per_capita = energy_service / pop

energy_service

<xarray.Dataset> Size: 4kB
Dimensions:   (region: 13, variable: 16, year: 2)
Coordinates:
  * variable  (variable) object 128B 'Energy Service|Transportation|Freight' ...
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Western Europe'
Data variables:
    value     (region, variable, year) float64 3kB 2.827e+03 4.541e+03 ... 6.282

In [11]:
# Define mapping from aggregated dataset variables to vehicle_targets variables
variable_mapping = {
    "Energy Service|Transportation|Freight": "Total freight. travel demand",
    "Energy Service|Transportation|Passenger": "Total pass. travel demand",
}

# Ensure matching dimensions for indexing
for source_var, target_var in variable_mapping.items():
    if source_var in energy_service_per_capita.variable:
        # Extract the data for the source variable as a DataArray
        data_array = xr_image_output_aggregated.sel(variable=source_var)
        print(data_array)
        print(dict(variable=target_var))

        # Assign the values from data_array to the target variable in vehicle_targets
        vehicle_targets.loc[dict(variable=target_var)] = data_array


<xarray.Dataset> Size: 476B
Dimensions:   (region: 13, year: 2)
Coordinates:
    variable  <U37 148B 'Energy Service|Transportation|Freight'
  * year      (year) int64 16B 2020 2060
  * region    (region) object 104B 'Canada' ... 'Western Europe'
Data variables:
    value     (region, year) float64 208B 2.827e+03 4.541e+03 ... 5e+04
{'variable': 'Total freight. travel demand'}


TypeError: cannot directly convert an xarray.Dataset into a numpy array. Instead, create an xarray.DataArray first, either with indexing on the Dataset or by invoking the `to_dataarray()` method.

In [ ]:
vehicle_targets.sel(Variable=target_var)

<xarray.DataArray (Time: 2, Region: 12)> Size: 192B
array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
Coordinates:
  * Time      (Time) int64 16B 2019 2060
  * Region    (Region) <U36 2kB 'Canada' ... 'Reforming EconomiesSubsaharan A...
    Variable  <U39 156B 'Total freight. travel demand'

In [ ]:
target_variables_narrow = {"Total pass. travel demand":"p-km/cap",
                    "Total freight. travel demand":"t-km/cap",
                    "passenger car":"Modal split - pkm/capita",
                    "passenger bus":"Modal split - pkm/capita",
                    "passenger aviation":"Modal split - pkm/capita",
                    "passenger walk":"Modal split - %",
                    "passenger bike":"Modal split - %",
                    "freight road heavy":"Modal split - tkm/capita",
                    "freight road light":"Modal split - tkm/capita",
                    "freight rail":"Modal split - tkm/capita",
                    "freight aviation":"Modal split - tkm/capita",
                    "Numb passengers per private vehicle": "Occupancy rate",
                    "Numb passengers per BUS": "Occupancy rate",
                    "Numb passengers per train": "Occupancy rate",
                    "Numb passengers per aviation passengers": "Occupancy rate",
	                "Ton per heavy freight": "Load rate",
                    "Ton per light freight": "Load rate",
                    "Ton per aviation freight": "Load rate",
                    "Ton per navigation freight": "Load rate"
}